In [1]:
import load_data as jtd
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter

import load_data as ld
import candidates as ds
import priority_queue as pq
import subgroups as s
import refinements as rf
import subgroup_selection as ss
import search as bs

# Data loading and transforming

In [2]:
df_library = pd.read_csv('algebrakit_firstbatch.csv')
collection_attributes = ['skillName', 'skillsTodo', 'stepType']

for colatt in collection_attributes:
    df_library[colatt] = df_library[colatt].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df_library = jtd.additional_columns(df_library)

df_library.head()

,exerciseID,sessionID,sessionSuccess,creationTimestamp,finished,marksEarned,marksTotal,skillName,stepType,errors,...,errorDesc_DivisionMultiplication_count,errorDesc_MultiplyAllTerms_count,errorDesc_SignError_count,errorDesc_SkippingTerms_count,errorDesc_ChangingTermType_count,errorDesc_AddedDenominator_count,errorDesc_MultiplicationAddition_count,errorTypes_MISCONCEPTION_count,errorTypes_MISTAKE_count,interactionStepsNeeded
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,2
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,0
2,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,4
3,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,4
4,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[],...,0,0,0,0,0,0,0,0,0,4


# Examples of functions and outputs

In [3]:
A = s.Subgroup("Failing", "{a1 = [8, 9]}", [1,3,4,6], 68.5, [35, 96, 94, 57])

beam = pq.PriorityQueue([A], 3)
print(beam._average_quality)

beam.summarize_queue() # average quality gets added after summarizing beam
print(beam._average_quality)

None
This queue has subgroups of average size 4.0 in a queue of length 1 and an average quality of 68.5.
68.5


In [4]:
rf.numerical_refinement(df_library, "marksTotal", 15)

[{'marksTotal': (1.0, 2.0)},
 {'marksTotal': (2.0, 3.0)},
 {'marksTotal': (3.0, 4.0)},
 {'marksTotal': (4.0, 5.0)},
 {'marksTotal': (5.0, 6.0)},
 {'marksTotal': (6.0, 7.0)},
 {'marksTotal': (7.0, 8.0)}]

In [5]:
rf.boolean_refinement(df_library, "finished")

[{'finished': np.False_}, {'finished': np.True_}]

In [6]:
cats = rf.categorical_refinement(df_library, "eventType")
att_list = []
for cas in cats:
    att_list.append(cas['eventType'])

att_list

[(1.0, 'EVALUATE'),
 (0.0, 'EVALUATE'),
 (1.0, 'HINT'),
 (0.0, 'HINT'),
 (1.0, 'GIVEUP'),
 (0.0, 'GIVEUP')]

In [7]:
rf.collection_refinement(df_library, "skillsTodo", 3)

[{'skillsTodo': ('FractionArithmeticMultiplicationProper',)},
 {'skillsTodo': ('FractionArithmeticSimplifyProper',)},
 {'skillsTodo': ('EquationsBalanceMethodDivideCoefficient',)},
 {'skillsTodo': ('EquationsOperationsMultiply',)},
 {'skillsTodo': ('EquationsBalanceMethodArrangeTerms',)},
 {'skillsTodo': ('EquationsOperationsAdd',)},
 {'skillsTodo': ('AlgebraAddingLikeTerms',)},
 {'skillsTodo': ('ExpandSingleBrackets',)},
 {'skillsTodo': ('FractionArithmeticSimilarImproperToMixed',)},
 {'skillsTodo': ('FractionArithmeticSimilarMixed',)},
 {'skillsTodo': ('FractionArithmeticSimilarMixedToImproper',)},
 {'skillsTodo': ('FractionArithmeticDivisionProper',)},
 {'skillsTodo': ('FractionArithmeticMultiplicationMixed',)},
 {'skillsTodo': ('FractionArithmeticAddProperUnlikeDenominator',)},
 {'skillsTodo': ('FractionArithmeticAddProperCommonDenominator',)},
 {'skillsTodo': ('FactorSingleBrackets',)},
 {'skillsTodo': ('FractionArithmeticMultiplicationProper',
   'FractionArithmeticSimplifyProper

# Setup for beam search

In [8]:
descriptive_space, target_space, attributes, descriptor_types = ds.split_spaces(df_library, 
                                                                                descriptors = [ 'skillName', 'stepType','questionType', 'itemType', 'interactionType',
                                                                                                'result.status', 'result.progress', 'result.scoring.penalties.marksPenalty',
                                                                                                'result.scoring.penalties.hintsRequested',
                                                                                                'result.scoring.penalties.mathErrors', 'result.scoring.usedAttempts',
                                                                                                'eventType', 'exerciseStatusAfterEvent', 'progressAfterEvent',
                                                                                                'inputStatus', 'skillsTodo', 
                                                                                                #self made below
                                                                                                'marksRatio', 'error_descriptions',
                                                                                                'error_types', 'positiveResultTags', 'negativeResultTags',
                                                                                                'step_CALCULATE_count', 'step_ALGEBRA_count', 'step_CONCEPT_count',
                                                                                                'step_STRATEGY_count', 'errorDesc_MultiplyVsPower_count',
                                                                                                'errorDesc_AddSubtract_count',
                                                                                                'errorDesc_EquivalentNumeratorDenominator_count',
                                                                                                'errorDesc_DivisionMultiplication_count',
                                                                                                'errorDesc_MultiplyAllTerms_count', 'errorDesc_SignError_count',
                                                                                                'errorDesc_SkippingTerms_count', 'errorDesc_ChangingTermType_count',
                                                                                                'errorDesc_AddedDenominator_count',
                                                                                                'errorDesc_MultiplicationAddition_count',
                                                                                                'errorTypes_MISCONCEPTION_count', 'errorTypes_MISTAKE_count',
                                                                                                'interactionStepsNeeded'], 
                                                                                targets = [ 'finished', 'marksEarned', 'marksTotal', 'result.scoring.marksEarned',
                                                                                            'result.scoring.marksTotal', 'result.scoring.failed', 'result.scoring.finished', 'result.progress'], # all to do with performance
                                                                                 ids = ['exerciseID', 'sessionID', 'creationTimestamp', 'questionID', 'interactionID', 'eventID'])
                                                                             #    skips=['errors', 'itemContent', 'interactionSolution', 'interactionDerivation', 'result.events',
                                                                             #           'result.tags', 'result.scoring.maxAttempts', 'eventTags', 'eventAnnotations']) # skips are to difficult atm
print(descriptor_types)
print("rerun")

{'categorical': ['result.status', 'eventType', 'exerciseStatusAfterEvent', 'inputStatus'], 'boolean': [], 'numerical': ['result.progress', 'result.scoring.penalties.marksPenalty', 'result.scoring.penalties.hintsRequested', 'result.scoring.penalties.mathErrors', 'result.scoring.usedAttempts', 'progressAfterEvent', 'marksRatio', 'step_CALCULATE_count', 'step_ALGEBRA_count', 'step_CONCEPT_count', 'step_STRATEGY_count', 'errorDesc_MultiplyVsPower_count', 'errorDesc_AddSubtract_count', 'errorDesc_EquivalentNumeratorDenominator_count', 'errorDesc_DivisionMultiplication_count', 'errorDesc_MultiplyAllTerms_count', 'errorDesc_SignError_count', 'errorDesc_SkippingTerms_count', 'errorDesc_ChangingTermType_count', 'errorDesc_AddedDenominator_count', 'errorDesc_MultiplicationAddition_count', 'errorTypes_MISCONCEPTION_count', 'errorTypes_MISTAKE_count', 'interactionStepsNeeded'], 'collection': ['skillName', 'stepType', 'skillsTodo', 'error_descriptions', 'error_types', 'positiveResultTags', 'negativ

In [9]:
candidates_per_attribute, candidates = ds.get_candidates_per_attribute(descriptive_space=descriptive_space, dtypes=descriptor_types, numerical_bins=3, collection_limit=3)
for att in candidates_per_attribute.keys():
    print(f'{att}: {candidates_per_attribute[att]}')

result.progress: [(0.0, 0.3333333333333333), (0.3333333333333333, 0.6666666666666666), (0.6666666666666666, 1.0)]
result.scoring.penalties.marksPenalty: [(0.0, 0.3333333333333333), (0.3333333333333333, 0.6666666666666666), (0.6666666666666666, 1.0)]
result.scoring.penalties.hintsRequested: [(0.0, 2.3333333333333335), (2.3333333333333335, 4.666666666666667), (4.666666666666667, 7.0)]
result.scoring.penalties.mathErrors: [(0.0, 3.0), (3.0, 6.0), (6.0, 9.0)]
result.scoring.usedAttempts: [(0.0, 3.0), (3.0, 6.0), (6.0, 9.0)]
progressAfterEvent: [(0.0, 0.3333333333333333), (0.3333333333333333, 0.6666666666666666), (0.6666666666666666, 1.0)]
marksRatio: [(0.0, 0.3333333333333333), (0.3333333333333333, 0.6666666666666666), (0.6666666666666666, 1.0)]
step_CALCULATE_count: [(0, 1.3333333333333333), (1.3333333333333333, 2.6666666666666665), (2.6666666666666665, 4)]
step_ALGEBRA_count: [(0, 0.6666666666666666), (0.6666666666666666, 1.3333333333333333), (1.3333333333333333, 2)]
step_CONCEPT_count: 

In [10]:
indices, new_descriptive_space = ss.check_record_inclusion(descriptive_space=descriptive_space, dtypes=descriptor_types, candidate=candidates[64])
new_descriptive_space

,exerciseID,sessionID,creationTimestamp,questionID,interactionID,eventID,skillName,stepType,questionType,itemType,...,errorDesc_DivisionMultiplication_count,errorDesc_MultiplyAllTerms_count,errorDesc_SignError_count,errorDesc_SkippingTerms_count,errorDesc_ChangingTermType_count,errorDesc_AddedDenominator_count,errorDesc_MultiplicationAddition_count,errorTypes_MISCONCEPTION_count,errorTypes_MISTAKE_count,interactionStepsNeeded
295,9f4a675e-c2b3-4e11-91cc-4445951bab60,34b46783-336e-49c1-b571-661fd725b251,1738837588,Q1,Ex8wE,1.738838e+12,"[FractionArithmeticMultiplicationProper, Numbe...","[CALCULATE, CONCEPT]",QUESTION,INTERACTION,...,0,0,0,0,0,0,0,1,0,2
296,9f4a675e-c2b3-4e11-91cc-4445951bab60,34b46783-336e-49c1-b571-661fd725b251,1738837588,Q1,NaN,NaN,"[FractionArithmeticMultiplicationProper, Numbe...","[CALCULATE, CONCEPT]",QUESTION,TEXT,...,0,0,0,0,0,0,0,1,0,0
297,9f4a675e-c2b3-4e11-91cc-4445951bab60,34b46783-336e-49c1-b571-661fd725b251,1738837588,Q2,gBslt,1.738838e+12,"[FractionArithmeticMultiplicationProper, Numbe...","[CALCULATE, CONCEPT]",QUESTION,INTERACTION,...,0,0,0,0,0,0,0,1,0,4
298,9f4a675e-c2b3-4e11-91cc-4445951bab60,34b46783-336e-49c1-b571-661fd725b251,1738837588,Q2,gBslt,1.738839e+12,"[FractionArithmeticMultiplicationProper, Numbe...","[CALCULATE, CONCEPT]",QUESTION,INTERACTION,...,0,0,0,0,0,0,0,1,0,4
299,9f4a675e-c2b3-4e11-91cc-4445951bab60,34b46783-336e-49c1-b571-661fd725b251,1738837588,Q2,gBslt,1.738839e+12,"[FractionArithmeticMultiplicationProper, Numbe...","[CALCULATE, CONCEPT]",QUESTION,INTERACTION,...,0,0,0,0,0,0,0,1,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20406,0beaa764-f558-4124-8d4d-b224ebdb2d4a,e9d25880-ebe8-4317-9b57-824754362557,1725370277,Q7,HuaWk,1.725371e+12,"[EquationsBalanceMethodArrangeTerms, Equations...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]",QUESTION,INTERACTION,...,0,0,0,0,0,0,0,1,0,2
20407,0beaa764-f558-4124-8d4d-b224ebdb2d4a,e9d25880-ebe8-4317-9b57-824754362557,1725370277,Q7,NaN,NaN,"[EquationsBalanceMethodArrangeTerms, Equations...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]",QUESTION,TEXT,...,0,0,0,0,0,0,0,1,0,0
20408,0beaa764-f558-4124-8d4d-b224ebdb2d4a,e9d25880-ebe8-4317-9b57-824754362557,1725370277,Q8,lwOZO,1.725371e+12,"[EquationsBalanceMethodArrangeTerms, Equations...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]",QUESTION,INTERACTION,...,0,0,0,0,0,0,0,1,0,2
20409,0beaa764-f558-4124-8d4d-b224ebdb2d4a,e9d25880-ebe8-4317-9b57-824754362557,1725370277,Q8,lwOZO,1.725371e+12,"[EquationsBalanceMethodArrangeTerms, Equations...","[STRATEGY, STRATEGY, ALGEBRA, STRATEGY, STRATEGY]",QUESTION,INTERACTION,...,0,0,0,0,0,0,0,1,0,2


In [11]:
ds.add_condition(candidates[89], candidates)

[{'exerciseStatusAfterEvent': (0.0, 'FINISHED'),
  'result.progress': (0.0, 0.3333333333333333)},
 {'exerciseStatusAfterEvent': (0.0, 'FINISHED'),
  'result.progress': (0.3333333333333333, 0.6666666666666666)},
 {'exerciseStatusAfterEvent': (0.0, 'FINISHED'),
  'result.progress': (0.6666666666666666, 1.0)},
 {'exerciseStatusAfterEvent': (0.0, 'FINISHED'),
  'result.scoring.penalties.marksPenalty': (0.0, 0.3333333333333333)},
 {'exerciseStatusAfterEvent': (0.0, 'FINISHED'),
  'result.scoring.penalties.marksPenalty': (0.3333333333333333,
   0.6666666666666666)},
 {'exerciseStatusAfterEvent': (0.0, 'FINISHED'),
  'result.scoring.penalties.marksPenalty': (0.6666666666666666, 1.0)},
 {'exerciseStatusAfterEvent': (0.0, 'FINISHED'),
  'result.scoring.penalties.hintsRequested': (0.0, 2.3333333333333335)},
 {'exerciseStatusAfterEvent': (0.0, 'FINISHED'),
  'result.scoring.penalties.hintsRequested': (2.3333333333333335,
   4.666666666666667)},
 {'exerciseStatusAfterEvent': (0.0, 'FINISHED'),
  '

# do beam search: first results, no anti redundancy
Problem: the data is skewed! So many sessions, and only a finish rate of 20% for the entire population!
- Maybe we should cover this by not computing the average finish rate for the subgroup and the population, but discover this per exercise rather than per session.
- Maybe even count per sessionID because of our data structuring -> failed sessions are tried more as thus are more represented here.
- So we make a subgroup description and evaluate this description based on a higher hierarchical quality: the average finish rate of the exercise in all recorded sessions.
- How do we eliminate "false attempts"?
- Entropy can also be based on number of exercises in the subgroup vs entire population, or sessions in subgroup.

In [12]:
descriptors = [ 'skillName', 'stepType','questionType', 'itemType', 'interactionType',
                'skillsTodo', #self made below
                 'error_descriptions', 'error_types', 'positiveResultTags', 'negativeResultTags', 'interactionStepsNeeded']
                
#                'step_CALCULATE_count', 
#               'step_ALGEBRA_count', 'step_CONCEPT_count', 'step_STRATEGY_count', 'errorDesc_MultiplyVsPower_count', 'errorDesc_AddSubtract_count', 
#                'errorDesc_EquivalentNumeratorDenominator_count', 'errorDesc_DivisionMultiplication_count', 'errorDesc_MultiplyAllTerms_count', 
#                'errorDesc_SignError_count', 'errorDesc_SkippingTerms_count', 'errorDesc_ChangingTermType_count', 'errorDesc_AddedDenominator_count', 
#                'errorDesc_MultiplicationAddition_count', 'errorTypes_MISCONCEPTION_count', 'errorTypes_MISTAKE_count', 'eventType',
#               'result.scoring.penalties.marksPenalty', 'result.scoring.penalties.hintsRequested', 'result.scoring.penalties.mathErrors', 'result.scoring.usedAttempts',] 

targets = [ 'finished', 'marksEarned', 'marksTotal', 'result.scoring.marksEarned', 'result.scoring.marksTotal', 'result.scoring.failed', 'result.scoring.finished', 'result.progress',
           'exerciseStatusAfterEvent', 'result.status', 'result.progress', 'progressAfterEvent',  'inputStatus', ] # all to do with performance

ids = ['exerciseID', 'sessionID', 'creationTimestamp', 'questionID', 'interactionID', 'eventID']

In [ ]:
# +- 12 mins
result_set = bs.beam_search(data=df_library, descriptors=descriptors, targets=targets, 
                            ids=ids, numerical_bins=3, collection_limit=3, 
                            k=10, max_depth=3, beam_width=20, model="exercise_finish_rate", minimum_size=25) #population = 20572, with repetitions

27.212142857142855
[24.41, 40.83, 59.3, 47.25, 28.87, 22.86, 23.95, 12.4, 33.33, 32.14, 8.06, 28.95, 4.07, 14.55]
start on level 1
start on level 2
start on level 3


In [14]:
for i in range(len(result_set._queue)):
    desc = result_set.get_descriptions()[i]
    avg = np.mean(result_set.get_values()[i])
    size = result_set.get_sizes()[i]

    print(i, desc, avg, size)

0 {'skillName': ('FractionArithmeticMultiplicationMixed',)} 0.0 43
1 {'skillName': ('ArithmeticOrderDivision',)} 0.0 36
2 {'skillName': ('FractionArithmeticSimilarMixedToImproper',)} 0.0 71
3 {'skillName': ('FractionArithmeticMultiplicationProper', 'AlgebraProductOfPowerFunctions')} 0.0 125
4 {'skillName': ('FractionArithmeticMultiplicationProper', 'NumberArithmeticMultiplicationInteger')} 0.0 77
5 {'skillName': ('AlgebraProductOfPowerFunctions', 'NumberArithmeticMultiplicationInteger')} 0.0 33
6 {'skillName': ('AlgebraProductOfPowerFunctions', 'NumberArithmeticMultiplicationDecimal')} 0.0 28
7 {'skillName': ('NumberArithmeticMultiplicationInteger', 'NumberArithmeticMultiplicationDecimal')} 0.0 44
8 {'skillName': ('NumberArithmeticMultiplicationInteger', 'FractionArithmeticSimplifyProper')} 0.0 90
9 {'skillName': ('NumberArithmeticMultiplicationInteger', 'NumberArithmeticAddConcept')} 0.0 35


In [15]:
result_set.get_descriptions()

[{'skillName': ('FractionArithmeticMultiplicationMixed',)},
 {'skillName': ('ArithmeticOrderDivision',)},
 {'skillName': ('FractionArithmeticSimilarMixedToImproper',)},
 {'skillName': ('FractionArithmeticMultiplicationProper',
   'AlgebraProductOfPowerFunctions')},
 {'skillName': ('FractionArithmeticMultiplicationProper',
   'NumberArithmeticMultiplicationInteger')},
 {'skillName': ('AlgebraProductOfPowerFunctions',
   'NumberArithmeticMultiplicationInteger')},
 {'skillName': ('AlgebraProductOfPowerFunctions',
   'NumberArithmeticMultiplicationDecimal')},
 {'skillName': ('NumberArithmeticMultiplicationInteger',
   'NumberArithmeticMultiplicationDecimal')},
 {'skillName': ('NumberArithmeticMultiplicationInteger',
   'FractionArithmeticSimplifyProper')},
 {'skillName': ('NumberArithmeticMultiplicationInteger',
   'NumberArithmeticAddConcept')}]

In [16]:
result_set.get_qualities()  

[np.float64(27.212142857142855),
 np.float64(27.212142857142855),
 np.float64(27.212142857142855),
 np.float64(27.212142857142855),
 np.float64(27.212142857142855),
 np.float64(27.212142857142855),
 np.float64(27.212142857142855),
 np.float64(27.212142857142855),
 np.float64(27.212142857142855),
 np.float64(27.212142857142855)]

In [17]:
result_set.get_sizes()

[43, 36, 71, 125, 77, 33, 28, 44, 90, 35]

In [18]:
for r in result_set.get_values():
    print(np.mean(r))

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [19]:
# finished rate of unique sessions per exercise ID
for ex in list(df_library['exerciseID'].unique()):
    only_ex = df_library[ df_library['exerciseID'] == ex ]
    total = len(list(only_ex['sessionID'].unique()))

    finished = only_ex[ only_ex['finished'] == True ]

    partial = len(list(finished['sessionID'].unique()))
    print(f"Exercise Id {ex} has a finish rate of {round(partial / total * 100, 2)}")

Exercise Id 9f4a675e-c2b3-4e11-91cc-4445951bab60 has a finish rate of 24.41
Exercise Id bb996be4-f9a9-48fb-9a0c-61e52a67fd69 has a finish rate of 40.83
Exercise Id a04508aa-09fa-4782-bfc3-effc3789fd47 has a finish rate of 59.3
Exercise Id 315bc927-cdd9-4d2a-8b3c-68f830f74e70 has a finish rate of 47.25
Exercise Id 486d7a7a-f282-48f4-8b74-c01a1c23a971 has a finish rate of 28.87
Exercise Id d4399f1b-009a-49c2-8d5f-b1ac805d195f has a finish rate of 22.86
Exercise Id 9aec3c1a-0100-40dd-a146-bdee6ff72fb3 has a finish rate of 23.95
Exercise Id ba15ab94-5da4-414e-a01d-107ea7c34d67 has a finish rate of 12.4
Exercise Id e3faf215-08e9-4ec0-b960-a35ccf261c7d has a finish rate of 33.33
Exercise Id 58b0759d-48ef-49f3-bb5f-7d999f35b12b has a finish rate of 32.14
Exercise Id 264d9ab3-57ba-4861-855b-fcc2a465c5db has a finish rate of 8.06
Exercise Id d93986a5-e93a-4be1-98cc-4d6ec991ef4f has a finish rate of 28.95
Exercise Id 086f5a82-c080-4d05-9431-03e992a0d7b3 has a finish rate of 4.07
Exercise Id 0bea

In [20]:
sorted([1, 2, 3, 4, 9, 5], reverse=True)

[9, 5, 4, 3, 2, 1]

In [46]:
df_library['eventType'][20411]

'EVALUATE'

In [43]:
df_library['result.tags'][20411]

[{'id': 'EquationsBalanceMethodArrangeTerms', 'weight': 100},
 {'id': 'EquationsOperationsAdd', 'weight': 100}]

In [47]:
df_library['eventType']

0        EVALUATE
1             NaN
2        EVALUATE
3        EVALUATE
4            HINT
           ...   
20567    EVALUATE
20568    EVALUATE
20569         NaN
20570    EVALUATE
20571         NaN
Name: eventType, Length: 20572, dtype: object

ModuleNotFoundError: No module named 'plotly'